# Pendulum Experiments: Cross-Morphology Transfer

**Goal:** Test if foundation models can transfer across systems with **different numbers of sensors**.

**Setup:**
- Train context: N=1,2,3 link pendulums (2, 4, 6 channels)
- Zero-shot test: N=5 link pendulum (10 channels - never seen!)

**This is the core Unify hypothesis:** Can we forecast on a system with more DOF than we've ever seen?

---

## N-Link Pendulum Illustration

```
N=1 (2 channels)     N=2 (4 channels)     N=3 (6 channels)     N=5 (10 channels)
                                                               
    O                    O                    O                    O
    |                    |                    |                    |
   [1] θ₁,θ̇₁           [1]                  [1]                  [1]
                         |                    |                    |
                        [2] θ₁,θ₂,θ̇₁,θ̇₂     [2]                  [2]
                                              |                    |
                                             [3] 6 obs            [3]
                                                                   |
                                                                  [4]
                                                                   |
                                                                  [5] 10 obs
```

Same physics (Lagrangian mechanics), different complexity!

In [ ]:
# Core imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
import json

# TabPFN
from tabpfn_time_series import TabPFNTSPipeline, TabPFNMode

pipeline = TabPFNTSPipeline(tabpfn_mode=TabPFNMode.CLIENT)
print("Setup complete!")

## Load Pendulum Data

In [ ]:
# Data paths
DATA_DIR = Path("data")

def load_pendulum_data(n_links, max_trajectories=100):
    """Load N-link pendulum trajectories."""
    folder = DATA_DIR / f"pendulum_n{n_links}"
    if not folder.exists():
        print(f"Warning: {folder} not found")
        return None, None
    
    # Load metadata
    with open(folder / "metadata.json") as f:
        metadata = json.load(f)
    
    # Load trajectories
    trajectories = []
    traj_files = sorted(folder.glob("trajectory_*.csv"))[:max_trajectories]
    for traj_file in traj_files:
        df = pd.read_csv(traj_file)
        df['trajectory_id'] = traj_file.stem
        trajectories.append(df)
    
    if trajectories:
        all_data = pd.concat(trajectories, ignore_index=True)
        return all_data, metadata
    return None, metadata

# Load all datasets
datasets = {}
for n in [1, 2, 3, 5]:
    data, meta = load_pendulum_data(n)
    if data is not None:
        datasets[n] = {'data': data, 'metadata': meta}
        n_traj = data['trajectory_id'].nunique()
        n_cols = len([c for c in data.columns if c.startswith('theta')])
        print(f"N={n}: {n_traj} trajectories, {n_cols} channels, {len(data)} total rows")
    else:
        print(f"N={n}: Not available yet")

In [ ]:
# Visualize sample trajectories
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for ax, n in zip(axes.flat, [1, 2, 3, 5]):
    if n not in datasets:
        ax.set_title(f'N={n}: Not available')
        continue
    
    data = datasets[n]['data']
    traj_id = data['trajectory_id'].unique()[0]  # First trajectory
    traj = data[data['trajectory_id'] == traj_id]
    
    # Plot all theta columns
    theta_cols = [c for c in traj.columns if c.startswith('theta_') and 'dot' not in c]
    for col in theta_cols:
        ax.plot(traj['time'], traj[col], label=col, lw=0.8)
    
    ax.set_title(f'N={n} Pendulum ({2*n} channels)')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Angle (rad)')
    ax.legend(loc='upper right', fontsize=8)

plt.suptitle('Sample Trajectories: N-Link Pendulums', y=1.02)
plt.tight_layout()
plt.show()

---
## Experiment 1: Same-N Forecasting Baseline

First, verify TabPFN works on pendulum data within the same N.

In [ ]:
def prepare_for_tabpfn(data, target_col, n_links):
    """Convert pendulum data to TabPFN format."""
    df = data.copy()
    
    # Create timestamp from time column
    df['timestamp'] = pd.to_datetime(df['time'], unit='s', origin='2024-01-01')
    df['item_id'] = df['trajectory_id']
    df['target'] = df[target_col]
    
    # Covariates: all other theta columns
    theta_cols = [c for c in df.columns if c.startswith('theta') and c != target_col]
    
    cols = ['item_id', 'timestamp', 'target'] + theta_cols
    return df[cols], theta_cols

# Test on N=2 (double pendulum)
if 2 in datasets:
    data_n2 = datasets[2]['data']
    
    # Use first trajectory
    traj = data_n2[data_n2['trajectory_id'] == data_n2['trajectory_id'].unique()[0]].copy()
    
    # Target: theta_1, Covariates: theta_2, theta_dot_1, theta_dot_2
    tabpfn_df, cov_cols = prepare_for_tabpfn(traj, 'theta_1', 2)
    
    print(f"Prepared data: {tabpfn_df.shape}")
    print(f"Target: theta_1, Covariates: {cov_cols}")
    tabpfn_df.head()

In [ ]:
# Same-N forecast: N=2 → N=2
if 2 in datasets:
    # Split: 80% context, predict next 50 timesteps
    split = int(len(tabpfn_df) * 0.8)
    pred_len = 50
    
    context = tabpfn_df.iloc[:split].copy()
    test = tabpfn_df.iloc[split:split+pred_len].copy()
    future = test.drop(columns=['target']).copy()
    
    print(f"Context: {len(context)} steps, Test: {len(test)} steps")
    
    # Forecast
    pred = pipeline.predict_df(context_df=context, future_df=future)
    
    actual = test['target'].values
    predicted = pred['target'].values
    mae_same = mean_absolute_error(actual, predicted)
    
    print(f"\nSame-N (N=2→N=2) MAE: {mae_same:.4f}")

In [ ]:
# Plot same-N forecast
if 2 in datasets:
    fig, ax = plt.subplots(figsize=(12, 4))
    
    ctx_plot = 100
    ax.plot(context['timestamp'].iloc[-ctx_plot:], context['target'].iloc[-ctx_plot:], 
            'b-', label='History', lw=1)
    ax.plot(test['timestamp'], actual, 'k-', label='Actual', lw=2)
    ax.plot(test['timestamp'], predicted, 'g--', label=f'Predicted (MAE={mae_same:.3f})', lw=2)
    ax.fill_between(test['timestamp'], pred[0.1].values, pred[0.9].values, 
                    alpha=0.2, color='green', label='80% CI')
    
    ax.axvline(context['timestamp'].iloc[-1], color='gray', ls=':')
    ax.set_title('Exp 1: Same-N Forecast (N=2 Double Pendulum)')
    ax.set_xlabel('Time'); ax.set_ylabel('θ₁ (rad)')
    ax.legend()
    plt.tight_layout(); plt.show()

---
## Experiment 2: Cross-N Transfer (The Key Test)

**Can we train on simpler systems and predict on more complex ones?**

- Context: N=1,2,3 pendulums
- Test: N=5 pendulum (never seen during training!)

In [ ]:
# THE FUNDAMENTAL CHALLENGE:
# N=1 has 2 channels: [theta_1, theta_dot_1]
# N=5 has 10 channels: [theta_1, ..., theta_5, theta_dot_1, ..., theta_dot_5]
#
# TabPFN requires SAME columns in context and future!
# This is exactly what Unify solves.

print("Cross-N Transfer Challenge:")
print("="*50)
for n in [1, 2, 3, 5]:
    if n in datasets:
        cols = [c for c in datasets[n]['data'].columns if c.startswith('theta')]
        print(f"N={n}: {len(cols)} channels - {cols}")

print("\n→ Cannot directly use N=1,2,3 context for N=5 prediction!")
print("→ This is the fixed-schema limitation we saw in ETTh1.")

In [ ]:
# THE CORE LIMITATION:
# To use TabPFN on N=5, we must use only the shared channels (theta_1)
# We LOSE all information from theta_2, theta_3, theta_4, theta_5!

if 5 in datasets:
    print("N=5 Pendulum: Information Loss Analysis")
    print("-"*60)

    data_n5 = datasets[5]['data']
    traj_n5 = data_n5[data_n5['trajectory_id'] == data_n5['trajectory_id'].unique()[0]].copy()

    # Prepare full N=5 data
    tabpfn_n5, cov_cols = prepare_for_tabpfn(traj_n5, 'theta_1', 5)
    print(f"N=5 has {len(cov_cols)} covariate channels: {cov_cols[:4]}...")

    # Split
    split = int(len(tabpfn_n5) * 0.8)
    pred_len = 50

    context_full = tabpfn_n5.iloc[:split].copy()
    test = tabpfn_n5.iloc[split:split+pred_len].copy()

    # OPTION A: Use ALL N=5 covariates (full information)
    future_full = test.drop(columns=['target']).copy()
    print(f"\nOption A (Full): Context {len(context_full)}, predicting {len(test)} steps with {len(cov_cols)} covariates")

    pred_full = pipeline.predict_df(context_df=context_full, future_df=future_full)
    mae_full = mean_absolute_error(test['target'].values, pred_full['target'].values)
    print(f"  MAE with ALL covariates: {mae_full:.4f}")

    # OPTION B: Use ONLY theta_1 (univariate - what we'd have from N=1 knowledge)
    context_uni = context_full[['item_id', 'timestamp', 'target']].copy()
    future_uni = test[['item_id', 'timestamp']].copy()  # No covariates!
    print(f"\nOption B (Univariate): Same trajectory, but NO covariates")

    pred_uni = pipeline.predict_df(context_df=context_uni, future_df=future_uni)
    mae_uni = mean_absolute_error(test['target'].values, pred_uni['target'].values)
    print(f"  MAE with NO covariates: {mae_uni:.4f}")

    # Store for plotting
    mae_cross = mae_uni  # For backward compatibility
    actual_n5 = test['target'].values
    predicted_n5 = pred_uni['target'].values
    pred_cross = pred_uni

    print(f"\n→ Information loss penalty: {(mae_uni/mae_full - 1)*100:+.1f}%")
    print(f"→ Using only theta_1 throws away {len(cov_cols)}/{len(cov_cols)+1} channels!")
else:
    print("N=5 data not yet available. Check back later!")

In [ ]:
# Plot N=5 information loss comparison
if 5 in datasets:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    test_ts = test['timestamp']
    
    # Left: Full information (all N=5 covariates)
    ax = axes[0]
    ax.plot(test_ts, test['target'].values, 'k-', label='Actual', lw=2)
    ax.plot(test_ts, pred_full['target'].values, 'g--', label=f'Predicted (MAE={mae_full:.4f})', lw=2)
    ax.fill_between(test_ts, pred_full[0.1].values, pred_full[0.9].values, alpha=0.2, color='green')
    ax.set_title(f'Full Information (9 covariates)')
    ax.legend(); ax.set_xlabel('Time'); ax.set_ylabel('θ₁ (rad)')
    
    # Right: Univariate (no covariates)
    ax = axes[1]
    ax.plot(test_ts, test['target'].values, 'k-', label='Actual', lw=2)
    ax.plot(test_ts, pred_uni['target'].values, 'r--', label=f'Predicted (MAE={mae_uni:.4f})', lw=2)
    ax.fill_between(test_ts, pred_uni[0.1].values, pred_uni[0.9].values, alpha=0.2, color='red')
    ax.set_title(f'Univariate Only (0 covariates)')
    ax.legend(); ax.set_xlabel('Time'); ax.set_ylabel('θ₁ (rad)')
    
    plt.suptitle('Exp 2: Information Loss in Fixed-Schema Models (N=5 Pendulum)', y=1.02)
    plt.tight_layout(); plt.show()
    
    print(f"\nKey Finding: Using covariates improves MAE by {(1 - mae_full/mae_uni)*100:.1f}%")
    print(f"But with fixed schema, we cannot leverage N=1,2,3 covariates for N=5!")

---
## Experiment 3: Complexity Scaling

Does prediction difficulty increase with system complexity (more links)?

- All systems use univariate prediction (theta_1 only)
- Same context/test split ratios
- Fair comparison across N=1,2,3,5

In [ ]:
# Exp 3: Compare forecasting difficulty across N
# Does complexity (more links) make prediction harder?

results = {}

for n in [1, 2, 3, 5]:
    if n not in datasets:
        continue
    
    data = datasets[n]['data']
    traj = data[data['trajectory_id'] == data['trajectory_id'].unique()[0]].copy()
    
    # Univariate prediction (theta_1 only - fair comparison across N)
    df, _ = prepare_for_tabpfn(traj, 'theta_1', n)
    df_uni = df[['item_id', 'timestamp', 'target']].copy()
    
    split = int(len(df_uni) * 0.8)
    context = df_uni.iloc[:split].copy()
    test = df_uni.iloc[split:split+50].copy()
    
    pred = pipeline.predict_df(context_df=context, prediction_length=50)
    mae = mean_absolute_error(test['target'].values, pred['target'].values)
    results[n] = mae
    print(f"N={n}: MAE = {mae:.4f}")

# Plot comparison
if results:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(results.keys(), results.values(), color=['#2ecc71', '#3498db', '#9b59b6', '#e74c3c'])
    ax.set_xlabel('Number of Links (N)')
    ax.set_ylabel('MAE (theta_1 prediction)')
    ax.set_title('Exp 3: Prediction Difficulty vs System Complexity')
    for n, mae in results.items():
        ax.text(n, mae + 0.01, f'{mae:.3f}', ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()
    
    print(f"\nObservation: More complex systems (higher N) may be harder to predict")
    print(f"because theta_1 motion is influenced by more interacting components.")

In [ ]:
# Final comparison table
print("\n" + "="*60)
print("RESULTS SUMMARY TABLE")
print("="*60)
print(f"{'Experiment':<35} {'MAE':<10} {'Notes'}")
print("-"*60)

if 2 in datasets:
    print(f"{'Exp 1: N=2 with covariates':<35} {mae_same:<10.4f} Baseline")
if 5 in datasets:
    print(f"{'Exp 2: N=5 with covariates':<35} {mae_full:<10.4f} Full info")
    print(f"{'Exp 2: N=5 univariate':<35} {mae_uni:<10.4f} Info loss: {(mae_uni/mae_full - 1)*100:+.1f}%")
for n, mae in results.items():
    print(f"{'Exp 3: N=' + str(n) + ' univariate':<35} {mae:<10.4f}")
print("="*60)

---
## The Unify Opportunity

**What we've shown:**
1. TabPFN works on same-N pendulum forecasting ✓
2. Cross-N transfer is LIMITED to shared channels only
3. Cannot use full sensor information from complex systems

**What Unify would enable:**
- Use ALL sensors from N=1,2,3 (2+4+6 = 12 channels total)
- Predict ALL sensors for N=5 (10 channels)
- Set-based encoder handles variable input dimensions
- Learn "universal pendulum physics" not "specific-N patterns"

In [ ]:
# Summary
print("="*70)
print("SUMMARY: Pendulum Cross-Morphology Experiments")
print("="*70)

if 2 in datasets:
    print(f"\nExp 1: Same-N Baseline (N=2→N=2)")
    print(f"  MAE: {mae_same:.4f}")
    print(f"  TabPFN works on pendulum data ✓")

if 5 in datasets:
    print(f"\nExp 2: Information Loss on N=5")
    print(f"  With ALL covariates:    MAE = {mae_full:.4f}")
    print(f"  With NO covariates:     MAE = {mae_uni:.4f}")
    print(f"  Performance drop: {(mae_uni/mae_full - 1)*100:+.1f}%")

print(f"\nTHE CROSS-MORPHOLOGY PROBLEM:")
print(f"  Training on N=1,2,3 teaches patterns about those systems' covariates")
print(f"  But N=5 has DIFFERENT covariate structure (10 vs 2,4,6 channels)")
print(f"  → Fixed-schema models cannot transfer learned covariate patterns!")
print(f"\nUNIFY SOLUTION:")
print(f"  Set-based encoder: treats each sensor as a token")
print(f"  Variable input: N=1→2 tokens, N=2→4 tokens, N=5→10 tokens")
print(f"  Shared latent: all systems map to same-dimension physics space")
print("="*70)

## Next Steps

1. **Implement Unify architecture:**
   - Set-based encoder (each sensor = token)
   - Fixed-dimension latent space
   - Permutation invariant aggregation

2. **Train on N=1,2,3 with FULL sensors:**
   - N=1: 2 tokens
   - N=2: 4 tokens
   - N=3: 6 tokens

3. **Zero-shot test on N=5:**
   - 10 tokens (never seen this many!)
   - Predict full trajectory